In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax as dfx
import optax
import numpy as np
from jax import random
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_PATH = 'spirals.npz'
TRAIN_SAMPLES = 10000
TEST_SAMPLES = 10000
USE_VALIDATION = False  # Enable/disable validation
VALIDATION_SPLIT = 0.2

INPUT_DIM = 3        # [x, y, time]
HIDDEN_DIM = 64
OUTPUT_DIM = 1       # Predicting single alpha value

NUM_EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
RANDOM_SEED = 42

OUTPUT_FILE = 'alpha_predictions.npy'

# ============================================================================
# DATA LOADING
# ============================================================================

def load_data(filepath, n_train=None, n_test=None):
    """Load spiral data and prepare for regression."""
    print("Loading data...")
    data = np.load(filepath)
    
    xy_train = data['xy_train'][:n_train] if n_train else data['xy_train']
    alpha_train = data['alpha_train'][:n_train] if n_train else data['alpha_train']
    xy_test = data['xy_test'][:n_test] if n_test else data['xy_test']
    
    # Create time dimension (normalized 0 to 1)
    seq_len = xy_train.shape[1]
    time = jnp.linspace(0, 1, seq_len)
    
    # Add time as third dimension: (batch, seq_len, 3)
    print("Adding time dimension...")
    def add_time(xy):
        batch_size = xy.shape[0]
        time_expanded = jnp.tile(time[None, :, None], (batch_size, 1, 1))
        return jnp.concatenate([xy, time_expanded], axis=-1)
    
    train_data = add_time(jnp.array(xy_train))
    test_data = add_time(jnp.array(xy_test))
    
    # Normalize x, y coordinates (not time, it's already 0-1)
    print("Normalizing x, y coordinates...")
    # Compute mean and std from training data only
    xy_mean = train_data[:, :, :2].mean(axis=(0, 1))  # Shape: (2,)
    xy_std = train_data[:, :, :2].std(axis=(0, 1))    # Shape: (2,)
    
    # Normalize x, y in both train and test
    train_data = train_data.at[:, :, :2].set((train_data[:, :, :2] - xy_mean) / (xy_std + 1e-8))
    test_data = test_data.at[:, :, :2].set((test_data[:, :, :2] - xy_mean) / (xy_std + 1e-8))
    
    print(f"Normalization stats - X: mean={xy_mean[0]:.4f}, std={xy_std[0]:.4f}")
    print(f"Normalization stats - Y: mean={xy_mean[1]:.4f}, std={xy_std[1]:.4f}")
    
    # Convert alpha to JAX arrays
    alpha_train = jnp.array(alpha_train).squeeze()
    
    print(f"Train shape: {train_data.shape}")
    print(f"Test shape: {test_data.shape}")
    print(f"Alpha shape: {alpha_train.shape}")
    print(f"Alpha range: [{alpha_train.min():.4f}, {alpha_train.max():.4f}]")
    
    return train_data, alpha_train, test_data

# ============================================================================
# GRU-ODE MODEL
# ============================================================================

class ODEFunc(eqx.Module):
    """ODE function for continuous evolution."""
    mlp: eqx.nn.MLP
    hidden_size: int

    def __init__(self, hidden_size: int, *, key):
        self.hidden_size = hidden_size
        self.mlp = eqx.nn.MLP(
            in_size=hidden_size,
            out_size=hidden_size,
            width_size=hidden_size * 2,
            depth=2,
            activation=jax.nn.softplus,
            key=key,
        )

    def __call__(self, t, h, args):
        h = jnp.reshape(h, (-1,))
        return self.mlp(h)


class GRUCell(eqx.Module):
    """GRU Cell for observation updates."""
    Wz: jnp.ndarray
    Wr: jnp.ndarray
    Wh: jnp.ndarray
    
    def __init__(self, input_size, hidden_size, key):
        key_z, key_r, key_h = random.split(key, 3)
        scale = 1.0 / jnp.sqrt(hidden_size)
        self.Wz = random.normal(key_z, (hidden_size + input_size, hidden_size)) * scale
        self.Wr = random.normal(key_r, (hidden_size + input_size, hidden_size)) * scale
        self.Wh = random.normal(key_h, (hidden_size + input_size, hidden_size)) * scale
    
    def __call__(self, x, h_prev):
        combined = jnp.concatenate([h_prev, x], axis=-1)
        z = jax.nn.sigmoid(combined @ self.Wz)
        r = jax.nn.sigmoid(combined @ self.Wr)
        combined_reset = jnp.concatenate([r * h_prev, x], axis=-1)
        h_prime = jnp.tanh(combined_reset @ self.Wh)
        h = (1 - z) * h_prime + z * h_prev
        return h


class GRUODERegressor(eqx.Module):
    """GRU-ODE for trajectory regression."""
    ode_func: ODEFunc
    gru_cell: GRUCell
    regressor: eqx.nn.Linear
    hidden_size: int

    def __init__(self, input_size: int, hidden_size: int, output_size: int, *, key):
        key_ode, key_gru, key_reg = random.split(key, 3)
        self.hidden_size = hidden_size
        self.ode_func = ODEFunc(hidden_size, key=key_ode)
        self.gru_cell = GRUCell(input_size, hidden_size, key=key_gru)
        self.regressor = eqx.nn.Linear(hidden_size, output_size, key=key_reg)

    def __call__(self, trajectory):
        """
        trajectory: (seq_len, 3) - [x, y, time]
        returns: scalar prediction for alpha
        """
        seq_len = trajectory.shape[0]
        h = jnp.zeros((self.hidden_size,), dtype=jnp.float32)
        
        # Extract time points
        times = trajectory[:, 2]
        
        solver = dfx.Dopri5()
        term = dfx.ODETerm(self.ode_func)
        
        for i in range(seq_len):
            # Evolve hidden state via ODE
            if i > 0:
                t0, t1 = times[i-1], times[i]
                solution = dfx.diffeqsolve(
                    term, solver,
                    t0=t0, t1=t1,
                    dt0=(t1 - t0) / 5.0,
                    y0=h,
                    saveat=dfx.SaveAt(t1=True),
                    max_steps=16,
                )
                h = jnp.reshape(solution.ys, (-1,))
            
            # Update with observation (x, y)
            obs = trajectory[i, :2]  # Just x, y
            h = self.gru_cell(obs, h)
        
        # Predict alpha from final hidden state
        alpha_pred = self.regressor(h)
        return alpha_pred.squeeze()  # Return scalar

# ============================================================================
# TRAINING
# ============================================================================

def loss_fn(model, trajectory, alpha_true):
    """Mean Squared Error loss."""
    alpha_pred = model(trajectory)
    return jnp.mean((alpha_pred - alpha_true) ** 2)

@eqx.filter_jit
def train_step(model, opt_state, trajectory, alpha_true, optimizer):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, trajectory, alpha_true)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

def compute_metrics(model, data, alphas):
    """Compute MAE and RMSE."""
    predictions = []
    for i in tqdm(range(len(data)), desc="Computing metrics", leave=False):
        pred = model(data[i])
        predictions.append(pred)
    predictions = jnp.array(predictions)
    
    mae = jnp.mean(jnp.abs(predictions - alphas))
    rmse = jnp.sqrt(jnp.mean((predictions - alphas) ** 2))
    return mae, rmse, predictions

def train_model(model, train_data, train_alphas, val_data, val_alphas, 
                num_epochs, batch_size, key, use_validation=False):
    """Training loop."""
    optimizer = optax.adam(LEARNING_RATE)
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))
    
    num_samples = train_data.shape[0]
    train_losses = []
    val_maes = []
    
    print("Starting training...")
    
    # Epoch progress bar
    for epoch in tqdm(range(num_epochs), desc="Epochs", position=0):
        key, subkey = random.split(key)
        perm = random.permutation(subkey, num_samples)
        
        epoch_loss = 0.0
        num_batches = 0
        
        # Batch progress bar
        batch_iterator = range(0, num_samples, batch_size)
        for i in tqdm(batch_iterator, desc=f"Epoch {epoch+1}/{num_epochs}", position=1, leave=False):
            batch_idx = perm[i:i+batch_size]
            batch_loss = 0.0
            
            for j in batch_idx:
                model, opt_state, loss = train_step(
                    model, opt_state, train_data[j], train_alphas[j], optimizer
                )
                batch_loss += loss
            
            batch_loss /= len(batch_idx)
            epoch_loss += batch_loss
            num_batches += 1
        
        avg_loss = epoch_loss / num_batches
        train_losses.append(float(avg_loss))
        
        # Validation metrics (optional)
        if use_validation:
            val_mae, val_rmse, _ = compute_metrics(model, val_data, val_alphas)
            val_maes.append(float(val_mae))
            tqdm.write(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}, Val MAE: {val_mae:.4f}, Val RMSE: {val_rmse:.4f}")
        else:
            tqdm.write(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")
    
    return model, train_losses, val_maes

# ============================================================================
# EVALUATION
# ============================================================================

@eqx.filter_jit
def predict_single(model, trajectory):
    return model(trajectory)

def predict_batch(model, data):
    predictions = []
    for i in tqdm(range(data.shape[0]), desc="Predicting"):
        pred = predict_single(model, data[i])
        predictions.append(pred)
    return jnp.array(predictions)

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    # Load data
    train_data, train_alphas, test_data = load_data(DATA_PATH, TRAIN_SAMPLES, TEST_SAMPLES)
    
    # Split validation (if enabled)
    if USE_VALIDATION:
        n_train = int(train_data.shape[0] * (1 - VALIDATION_SPLIT))
        val_data = train_data[n_train:]
        val_alphas = train_alphas[n_train:]
        train_data = train_data[:n_train]
        train_alphas = train_alphas[:n_train]
        
        print(f"\nTraining samples: {len(train_data)}")
        print(f"Validation samples: {len(val_data)}")
    else:
        val_data = None
        val_alphas = None
        print(f"\nTraining samples: {len(train_data)}")
        print("Validation: DISABLED")
    
    # Initialize model
    key = random.PRNGKey(RANDOM_SEED)
    key, model_key = random.split(key)
    
    model = GRUODERegressor(
        input_size=2,  # x, y (time is used for ODE evolution)
        hidden_size=HIDDEN_DIM,
        output_size=OUTPUT_DIM,
        key=model_key
    )
    
    # Train
    model, train_losses, val_maes = train_model(
        model, train_data, train_alphas, val_data, val_alphas,
        NUM_EPOCHS, BATCH_SIZE, key, USE_VALIDATION
    )
    
    print("\nTraining complete!")
    
    # Evaluate on training set
    train_mae, train_rmse, train_predictions = compute_metrics(model, train_data, train_alphas)
    
    print(f"\nTraining Set Performance:")
    print(f"MAE: {train_mae:.4f}")
    print(f"RMSE: {train_rmse:.4f}")
    
    # Validation performance (if enabled)
    if USE_VALIDATION:
        val_mae, val_rmse, val_predictions = compute_metrics(model, val_data, val_alphas)
        
        print(f"\nValidation Set Performance:")
        print(f"MAE: {val_mae:.4f}")
        print(f"RMSE: {val_rmse:.4f}")
    
    # Test predictions
    test_predictions = predict_batch(model, test_data)
    
    # Save as (TEST_SAMPLES, 1) shape
    test_predictions_reshaped = np.array(test_predictions).reshape(-1, 1)
    np.save(OUTPUT_FILE, test_predictions_reshaped)
    print(f"\nTest predictions saved to {OUTPUT_FILE}")
    print(f"Shape: {test_predictions_reshaped.shape}")
    
    # Plotting
    if USE_VALIDATION:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Training loss
        axes[0, 0].plot(train_losses)
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('MSE Loss')
        axes[0, 0].set_title('Training Loss')
        axes[0, 0].grid(True)
        
        # Validation MAE
        axes[0, 1].plot(val_maes)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('MAE')
        axes[0, 1].set_title('Validation MAE')
        axes[0, 1].grid(True)
        
        # Training predictions vs actual
        axes[1, 0].scatter(train_alphas, train_predictions, alpha=0.5, s=20)
        min_val = min(train_alphas.min(), train_predictions.min())
        max_val = max(train_alphas.max(), train_predictions.max())
        axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
        axes[1, 0].set_xlabel('True Alpha')
        axes[1, 0].set_ylabel('Predicted Alpha')
        axes[1, 0].set_title(f'Training Set (MAE: {train_mae:.4f}, RMSE: {train_rmse:.4f})')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
        
        # Validation predictions vs actual
        axes[1, 1].scatter(val_alphas, val_predictions, alpha=0.5, s=20)
        min_val = min(val_alphas.min(), val_predictions.min())
        max_val = max(val_alphas.max(), val_predictions.max())
        axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
        axes[1, 1].set_xlabel('True Alpha')
        axes[1, 1].set_ylabel('Predicted Alpha')
        axes[1, 1].set_title(f'Validation Set (MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f})')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
        
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Training loss
        axes[0].plot(train_losses)
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('MSE Loss')
        axes[0].set_title('Training Loss')
        axes[0].grid(True)
        
        # Training predictions vs actual
        axes[1].scatter(train_alphas, train_predictions, alpha=0.5, s=20)
        min_val = min(train_alphas.min(), train_predictions.min())
        max_val = max(train_alphas.max(), train_predictions.max())
        axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
        axes[1].set_xlabel('True Alpha')
        axes[1].set_ylabel('Predicted Alpha')
        axes[1].set_title(f'Training Set (MAE: {train_mae:.4f}, RMSE: {train_rmse:.4f})')
        axes[1].legend()
        axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_results.png', dpi=150)
    print("Plot saved to training_results.png")
    plt.show()

Assignment 3: GRU-ODE for Spiral Alpha Prediction

Processing 100% dataset

Loading spirals_100.npz...
Train shape: (1000, 100, 3)
Validation shape: (1000, 100, 3)
Test shape: (1000, 100, 3)
Alpha train shape: (1000, 1)
Normalizing coordinates...
Normalization - X: mean=0.0022, std=13.2288
Normalization - Y: mean=-0.0428, std=13.0091
Alpha range: [-17.6583, 12.0299]
Alpha - mean=0.0571, std=2.4921
Starting training...


Epochs:   0%|          | 0/5 [08:38<?, ?it/s]


KeyboardInterrupt: 